# Local Outlier Factor (LOF)

Density-based anomaly detector: flags a window if its local neighbourhood density is much lower than its neighbours' — i.e. it sits in a sparse region relative to normal data. Uses `novelty=True` so it can score new (test/fault) windows.

Trained STRICTLY on `normal` sessions only — motor_off and transition sessions are excluded entirely, so this model's job is simply: does this look like normal running, or not.

In [ ]:
# ==========================================================
# CELL 0 — install dependencies
# ==========================================================
%pip install pandas numpy scikit-learn joblib openpyxl matplotlib skl2onnx onnx onnxruntime


In [ ]:
# ==========================================================
# CELL 1 — imports
# ==========================================================
import os
import glob
import random
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import MinMaxScaler

from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType


In [ ]:
# ==========================================================
# CELL 2 — configuration
# ==========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_PATH = r"../../data/Motor_data_model"
SCALE_DIVISOR = 1_000_000.0

FEATURE_COLUMNS = [
    "T_slope_10s",
    "PWM",
    "Irms",
    "Vrms",
    "Current_Variance",
    "Current_Crest_Factor",
    "Vib_RMS",
    "Vib_Min",
    "Vib_Q1",
    "Vib_Median",
    "Vib_Q3",
    "Vib_Max",
    "Vib_Variance",
    "Vib_Skewness",
]

N_FEATURES = len(FEATURE_COLUMNS)
WINDOW_LEN = 30
WINDOW_STEP = 10
ROW_PERIOD_SEC = 0.015

N_SPLITS = 5
N_NEIGHBORS = 20
SCORE_PERCENTILE = 99.0
FILE_ANOMALY_FRACTION_THRESHOLD = 0.10

OUTPUT_DIR = "pure_normal_lof_output"
CV_DIR = os.path.join(OUTPUT_DIR, "cross_validation")
FINAL_DIR = os.path.join(OUTPUT_DIR, "final_model")
FINAL_EVAL_DIR = os.path.join(OUTPUT_DIR, "final_model_evaluation")
MANUAL_TEST_DIR = os.path.join(OUTPUT_DIR, "manual_test_results")

for folder in [CV_DIR, FINAL_DIR, FINAL_EVAL_DIR, MANUAL_TEST_DIR]:
    os.makedirs(folder, exist_ok=True)

In [ ]:
# ==========================================================
# CELL 3 — detect normal and anomaly files
# ==========================================================
def normalized_path(path):
    return str(path).replace("\\", "/").lower()


def normal_state_for(path):
    path_text = normalized_path(path)
    stem = Path(path).stem.lower()

    excluded = [
        "motor_off",
        "motor_being_on",
        "motor_being_off",
        "transition",
        "off_on_spikes",
    ]

    if any(word in path_text for word in excluded):
        return None

    if "/normal/" in path_text or stem.startswith("normal_") or stem == "normal":
        return "normal_running"

    return None


def anomaly_label_for(path):
    path_text = normalized_path(path)

    if "motor_being_on" in path_text:
        return "motor_being_on"
    if "motor_being_off" in path_text:
        return "motor_being_off"
    if "/motor_off/" in path_text or "motor_off" in Path(path_text).stem:
        return "motor_off"
    if "electrical fault" in path_text or "abnormal_current" in path_text:
        return "electrical_fault"
    if "mechanical fault" in path_text or "abnormal_vibration" in path_text:
        return "mechanical_fault"
    if "thermal fault" in path_text or "abnormal_temp" in path_text:
        return "thermal_fault"

    return None


In [ ]:
# ==========================================================
# CELL 4 — discover and read dataset files
# ==========================================================
def discover_files(base_path):
    files = glob.glob(
        os.path.join(base_path, "**", "*.xlsx"),
        recursive=True,
    )

    files += glob.glob(
        os.path.join(base_path, "**", "*.csv"),
        recursive=True,
    )

    return sorted(set(files))


def read_one(path):
    path = Path(path)

    if path.suffix.lower() == ".xlsx":
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path)

    missing = [
        column
        for column in FEATURE_COLUMNS
        if column not in df.columns
    ]

    if missing:
        raise ValueError(
            f"Missing columns in {path.name}: {missing}"
        )

    clean = df[FEATURE_COLUMNS].apply(
        pd.to_numeric,
        errors="coerce",
    )

    clean = clean / SCALE_DIVISOR
    clean = clean.replace([np.inf, -np.inf], np.nan)
    clean = clean.dropna().reset_index(drop=True)

    if len(clean) < WINDOW_LEN:
        raise ValueError(
            f"{path.name} has only {len(clean)} valid rows."
        )

    return clean

In [ ]:
# ==========================================================
# CELL 5 — load normal-with-load and anomaly sessions
# ==========================================================
files = discover_files(BASE_PATH)

if not files:
    raise RuntimeError(
        f"No CSV or Excel files found under: {BASE_PATH}"
    )


def is_normal_with_load(path):
    """
    Return True only when the normal file contains loaded-motor data.

    Accepted examples:
        normal_load_id1.xlsx
        normal_with_load.xlsx
        normal_loaded.xlsx
        Normal With Load/file.xlsx

    Rejected examples:
        normal_without_load.xlsx
        normal_no_load.xlsx
        normal_unloaded.xlsx
    """
    path_text = normalized_path(path)

    # Make filename/folder naming styles comparable.
    path_text = (
        path_text
        .replace(" ", "_")
        .replace("-", "_")
    )

    without_load_markers = [
        "without_load",
        "no_load",
        "withoutload",
        "noload",
        "unloaded",
    ]

    if any(
        marker in path_text
        for marker in without_load_markers
    ):
        return False

    with_load_markers = [
        "with_load",
        "withload",
        "normal_load",
        "normal_loaded",
    ]

    return any(
        marker in path_text
        for marker in with_load_markers
    )


healthy_sessions = []
fault_sessions = []
excluded_normal_files = []

for file_path in files:
    normal_state = normal_state_for(file_path)
    fault_label = anomaly_label_for(file_path)

    if normal_state is None and fault_label is None:
        continue

    # A normal file is accepted only if it is a with-load file.
    if (
        normal_state is not None
        and not is_normal_with_load(file_path)
    ):
        excluded_normal_files.append(
            os.path.relpath(file_path, BASE_PATH)
        )
        continue

    try:
        df = read_one(file_path)

    except Exception as error:
        print(
            "SKIP:",
            file_path,
            "|",
            error,
        )
        continue

    record = {
        "X": df.to_numpy(dtype=np.float32),
        "file": os.path.relpath(
            file_path,
            BASE_PATH,
        ),
        "rows": len(df),
    }

    if normal_state is not None:
        record.update({
            "state": normal_state,
            "load_condition": "with_load",
            "binary_label": 0,
        })

        healthy_sessions.append(record)

    else:
        record.update({
            "fault_label": fault_label,
            "binary_label": 1,
        })

        fault_sessions.append(record)


if not healthy_sessions:
    raise RuntimeError(
        "No normal WITH-LOAD files were loaded. "
        "Check the filenames and folders."
    )

if not fault_sessions:
    raise RuntimeError(
        "No anomaly files were loaded."
    )


print("\nNormal WITH-LOAD files used by the model:")
for session in healthy_sessions:
    print("  USED:", session["file"])

print("\nNormal WITHOUT-LOAD files excluded:")
for file_name in excluded_normal_files:
    print("  EXCLUDED:", file_name)

print("\nDATASET SUMMARY")
print("Normal with-load files:", len(healthy_sessions))
print("Normal files excluded:", len(excluded_normal_files))
print("Anomaly files:", len(fault_sessions))

In [ ]:
# ==========================================================
# CELL 6 — create windows and flatten input
# ==========================================================
def make_windows_from_array(X_raw, scaler):
    X_scaled = scaler.transform(X_raw).astype(np.float32)

    starts = np.arange(
        0,
        len(X_scaled) - WINDOW_LEN + 1,
        WINDOW_STEP,
        dtype=np.int64,
    )

    if len(starts) == 0:
        return None, None

    windows = np.stack([
        X_scaled[start:start + WINDOW_LEN]
        for start in starts
    ]).astype(np.float32)

    return windows, starts


def build_windows_from_sessions(sessions, scaler):
    all_windows = []

    for session in sessions:
        windows, _ = make_windows_from_array(
            session["X"],
            scaler,
        )

        if windows is not None:
            all_windows.append(windows)

    if not all_windows:
        return np.empty(
            (0, WINDOW_LEN, N_FEATURES),
            dtype=np.float32,
        )

    return np.concatenate(
        all_windows,
        axis=0,
    )


def flatten_windows(X):
    return X.reshape(len(X), -1)


In [ ]:
# ==========================================================
# CELL 7 — build LOF model and anomaly score
# ==========================================================
def build_lof_model(number_of_samples):
    neighbors = min(
        N_NEIGHBORS,
        max(2, number_of_samples - 1),
    )

    return LocalOutlierFactor(
        n_neighbors=neighbors,
        novelty=True,
    )


def anomaly_score(model, X):
    scores = model.score_samples(
        flatten_windows(X)
    )

    return (-scores).astype(np.float32)


def calibrate_threshold(model, X_normal):
    scores = anomaly_score(model, X_normal)

    return float(
        np.percentile(
            scores,
            SCORE_PERCENTILE,
        )
    )


In [ ]:
# ==========================================================
# CELL 8 — create file-level and window-level predictions
# ==========================================================
def aggregate_predictions(
    sessions,
    model,
    scaler,
    score_threshold,
):
    file_rows = []
    window_rows = []

    for session in sessions:
        windows, starts = make_windows_from_array(
            session["X"],
            scaler,
        )

        if windows is None:
            continue

        scores = anomaly_score(model, windows)
        flags = (scores > score_threshold).astype(np.int64)

        state = session.get(
            "state",
            session.get("fault_label", "unknown"),
        )

        true_label = int(session["binary_label"])
        anomaly_fraction = float(np.mean(flags))

        file_prediction = int(
            anomaly_fraction
            >= FILE_ANOMALY_FRACTION_THRESHOLD
        )

        file_rows.append({
            "file": session["file"],
            "state": state,
            "true_label": true_label,
            "pred_label": file_prediction,
            "windows": len(windows),
            "anomaly_windows": int(np.sum(flags)),
            "anomaly_fraction": anomaly_fraction,
            "mean_score": float(np.mean(scores)),
            "max_score": float(np.max(scores)),
            "score_threshold": score_threshold,
        })

        for window_id in range(len(windows)):
            window_rows.append({
                "file": session["file"],
                "state": state,
                "window": window_id,
                "start_row": int(starts[window_id]),
                "true_label": true_label,
                "pred_label": int(flags[window_id]),
                "anomaly_score": float(scores[window_id]),
                "score_threshold": score_threshold,
            })

    return (
        pd.DataFrame(file_rows),
        pd.DataFrame(window_rows),
    )



In [ ]:
# ==========================================================
# CELL 9 — build file-wise cross-validation table
# ==========================================================
all_sessions = healthy_sessions + fault_sessions

cv_table = pd.DataFrame({
    "label": [
        session["binary_label"]
        for session in all_sessions
    ],
    "group": [
        session["file"]
        for session in all_sessions
    ],
})

In [ ]:
# ==========================================================
# CELL 10 — five-fold file-wise cross-validation
# ==========================================================
splitter = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

cv_metrics = []
cv_file_results = []
cv_window_results = []

for fold, (train_index, test_index) in enumerate(
    splitter.split(
        cv_table,
        cv_table["label"],
        groups=cv_table["group"],
    ),
    start=1,
):
    train_sessions = [
        all_sessions[index]
        for index in train_index
    ]

    test_sessions = [
        all_sessions[index]
        for index in test_index
    ]

    normal_train_sessions = [
        session
        for session in train_sessions
        if session["binary_label"] == 0
    ]

    rng = np.random.default_rng(SEED + fold)
    shuffled_ids = rng.permutation(
        len(normal_train_sessions)
    )

    number_calibration_files = max(
        1,
        round(0.20 * len(shuffled_ids)),
    )

    calibration_ids = set(
        shuffled_ids[:number_calibration_files]
    )

    fit_sessions = [
        session
        for index, session in enumerate(normal_train_sessions)
        if index not in calibration_ids
    ]

    calibration_sessions = [
        session
        for index, session in enumerate(normal_train_sessions)
        if index in calibration_ids
    ]

    scaler = MinMaxScaler(
        feature_range=(0, 1)
    )

    scaler.fit(
        np.concatenate([
            session["X"]
            for session in fit_sessions
        ])
    )

    X_fit = build_windows_from_sessions(
        fit_sessions,
        scaler,
    )

    X_calibration = build_windows_from_sessions(
        calibration_sessions,
        scaler,
    )

    model = build_lof_model(len(X_fit))
    model.fit(flatten_windows(X_fit))

    score_threshold = calibrate_threshold(
        model,
        X_calibration,
    )

    fold_file_df, fold_window_df = aggregate_predictions(
        test_sessions,
        model,
        scaler,
        score_threshold,
    )

    fold_file_df.insert(0, "fold", fold)
    fold_window_df.insert(0, "fold", fold)

    cv_file_results.append(fold_file_df)
    cv_window_results.append(fold_window_df)

    y_true = fold_window_df["true_label"].to_numpy()
    y_pred = fold_window_df["pred_label"].to_numpy()
    y_score = fold_window_df["anomaly_score"].to_numpy()

    fold_metrics = {
        "fold": fold,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            y_pred,
        ),
        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "recall": recall_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "f1": f1_score(
            y_true,
            y_pred,
            zero_division=0,
        ),
        "roc_auc": (
            roc_auc_score(y_true, y_score)
            if len(np.unique(y_true)) == 2
            else np.nan
        ),
        "score_threshold": score_threshold,
    }

    cv_metrics.append(fold_metrics)

    print("\nFOLD", fold)
    print(pd.Series(fold_metrics))


In [ ]:
# ==========================================================
# CELL 11 — combined cross-validation results
# ==========================================================
cv_metrics_df = pd.DataFrame(cv_metrics)

cv_file_results_df = pd.concat(
    cv_file_results,
    ignore_index=True,
)

cv_window_results_df = pd.concat(
    cv_window_results,
    ignore_index=True,
)

cv_metrics_df.to_csv(
    os.path.join(CV_DIR, "cv_metrics.csv"),
    index=False,
)

cv_file_results_df.to_csv(
    os.path.join(CV_DIR, "cv_file_predictions.csv"),
    index=False,
)

cv_window_results_df.to_csv(
    os.path.join(CV_DIR, "cv_window_predictions.csv"),
    index=False,
)

cv_y_true = cv_window_results_df[
    "true_label"
].to_numpy()

cv_y_pred = cv_window_results_df[
    "pred_label"
].to_numpy()

cv_y_score = cv_window_results_df[
    "anomaly_score"
].to_numpy()

overall_cv_metrics = {
    "accuracy": accuracy_score(
        cv_y_true,
        cv_y_pred,
    ),
    "balanced_accuracy": balanced_accuracy_score(
        cv_y_true,
        cv_y_pred,
    ),
    "precision": precision_score(
        cv_y_true,
        cv_y_pred,
        zero_division=0,
    ),
    "recall": recall_score(
        cv_y_true,
        cv_y_pred,
        zero_division=0,
    ),
    "f1": f1_score(
        cv_y_true,
        cv_y_pred,
        zero_division=0,
    ),
    "roc_auc": roc_auc_score(
        cv_y_true,
        cv_y_score,
    ),
}

print("\nOVERALL CV RESULTS")
print(pd.Series(overall_cv_metrics))

print(classification_report(
    cv_y_true,
    cv_y_pred,
    target_names=["normal", "anomaly"],
    zero_division=0,
))

ConfusionMatrixDisplay(
    confusion_matrix(
        cv_y_true,
        cv_y_pred,
        labels=[0, 1],
    ),
    display_labels=[
        "normal window",
        "anomaly window",
    ],
).plot(values_format="d")

plt.tight_layout()
plt.savefig(
    os.path.join(CV_DIR, "confusion_matrix.png"),
    dpi=180,
)
plt.show()


In [ ]:
# ==========================================================
# CELL 12 — train final LOF model on all normal data
# ==========================================================
final_scaler = MinMaxScaler(
    feature_range=(0, 1)
)

final_scaler.fit(
    np.concatenate([
        session["X"]
        for session in healthy_sessions
    ])
)

X_all_normal = build_windows_from_sessions(
    healthy_sessions,
    final_scaler,
)

final_model = build_lof_model(
    len(X_all_normal)
)

final_model.fit(
    flatten_windows(X_all_normal)
)

FINAL_SCORE_THRESHOLD = calibrate_threshold(
    final_model,
    X_all_normal,
)

print("Final score threshold:", FINAL_SCORE_THRESHOLD)

In [ ]:
# ==========================================================
# CELL 13 — final model evaluation
# ==========================================================
rng = np.random.default_rng(SEED)

number_normal_evaluation_files = max(
    1,
    round(0.20 * len(healthy_sessions)),
)

normal_ids = rng.choice(
    len(healthy_sessions),
    size=number_normal_evaluation_files,
    replace=False,
)

selected_normal_sessions = [
    healthy_sessions[int(index)]
    for index in normal_ids
]

fault_groups = {}

for session in fault_sessions:
    fault_groups.setdefault(
        session["fault_label"],
        [],
    ).append(session)

selected_fault_sessions = [
    group[int(rng.integers(len(group)))]
    for group in fault_groups.values()
]

evaluation_sessions = (
    selected_normal_sessions
    + selected_fault_sessions
)

final_file_df, final_window_df = aggregate_predictions(
    evaluation_sessions,
    final_model,
    final_scaler,
    FINAL_SCORE_THRESHOLD,
)

final_y_true = final_window_df[
    "true_label"
].to_numpy()

final_y_pred = final_window_df[
    "pred_label"
].to_numpy()

final_y_score = final_window_df[
    "anomaly_score"
].to_numpy()

final_metrics = {
    "accuracy": accuracy_score(
        final_y_true,
        final_y_pred,
    ),
    "balanced_accuracy": balanced_accuracy_score(
        final_y_true,
        final_y_pred,
    ),
    "normal_accuracy": float(
        np.mean(
            final_y_pred[final_y_true == 0] == 0
        )
    ),
    "anomaly_accuracy": float(
        np.mean(
            final_y_pred[final_y_true == 1] == 1
        )
    ),
    "precision": precision_score(
        final_y_true,
        final_y_pred,
        zero_division=0,
    ),
    "recall": recall_score(
        final_y_true,
        final_y_pred,
        zero_division=0,
    ),
    "f1": f1_score(
        final_y_true,
        final_y_pred,
        zero_division=0,
    ),
    "roc_auc": roc_auc_score(
        final_y_true,
        final_y_score,
    ),
    "score_threshold": FINAL_SCORE_THRESHOLD,
}

print("\nFINAL RESULTS")
print(pd.Series(final_metrics))

print(classification_report(
    final_y_true,
    final_y_pred,
    target_names=["normal", "anomaly"],
    zero_division=0,
))

final_file_df.to_csv(
    os.path.join(
        FINAL_EVAL_DIR,
        "file_predictions.csv",
    ),
    index=False,
)

final_window_df.to_csv(
    os.path.join(
        FINAL_EVAL_DIR,
        "window_predictions.csv",
    ),
    index=False,
)

pd.DataFrame([final_metrics]).to_csv(
    os.path.join(
        FINAL_EVAL_DIR,
        "metrics.csv",
    ),
    index=False,
)

In [ ]:
# ==========================================================
# CELL 14 — save final LOF model, scaler, and threshold
# ==========================================================
MODEL_PATH = os.path.join(
    FINAL_DIR,
    "pure_normal_lof.pkl",
)

SCALER_PATH = os.path.join(
    FINAL_DIR,
    "pure_normal_scaler.pkl",
)

THRESHOLD_PATH = os.path.join(
    FINAL_DIR,
    "lof_threshold.npz",
)

joblib.dump(
    final_model,
    MODEL_PATH,
)

joblib.dump(
    final_scaler,
    SCALER_PATH,
)

np.savez(
    THRESHOLD_PATH,
    score_threshold=np.float32(
        FINAL_SCORE_THRESHOLD
    ),
    file_anomaly_fraction_threshold=np.float32(
        FILE_ANOMALY_FRACTION_THRESHOLD
    ),
)

print("Saved:", MODEL_PATH)
print("Saved:", SCALER_PATH)
print("Saved:", THRESHOLD_PATH)


In [ ]:
# ==========================================================
# CELL 14.1 — export LOF model to ONNX
# ==========================================================


LOF_ONNX_PATH = os.path.join(
    FINAL_DIR,
    "pure_normal_lof.onnx",
)

initial_type = [
    (
        "scaled_window",
        FloatTensorType([
            None,
            WINDOW_LEN * N_FEATURES,
        ]),
    )
]

lof_onnx_model = convert_sklearn(
    final_model,
    initial_types=initial_type,
    target_opset=21,
)

with open(LOF_ONNX_PATH, "wb") as file:
    file.write(
        lof_onnx_model.SerializeToString()
    )

print("Saved:", LOF_ONNX_PATH)
print(
    "ONNX input shape:",
    f"[batch, {WINDOW_LEN * N_FEATURES}]",
)
print(
    "Important: ONNX input must already be",
    "MinMax-scaled and flattened.",
)


In [ ]:


# ==========================================================
# CELL 14.2 — export complete LOF model as a C header
# ==========================================================
LOF_HEADER_PATH = os.path.join(
    FINAL_DIR,
    "pure_normal_lof_model.h",
)

training_windows = np.asarray(
    final_model._fit_X,
    dtype=np.float32,
)

training_lrd = np.asarray(
    final_model._lrd,
    dtype=np.float32,
)

training_k_distance = np.asarray(
    final_model._distances_fit_X_[:, -1],
    dtype=np.float32,
)

scaler_min = np.asarray(
    final_scaler.data_min_,
    dtype=np.float32,
)

scaler_max = np.asarray(
    final_scaler.data_max_,
    dtype=np.float32,
)

number_training_windows = training_windows.shape[0]
number_inputs = training_windows.shape[1]
number_neighbors = int(final_model.n_neighbors_)


def write_float_array(file, name, values, values_per_line=8):
    values = np.asarray(
        values,
        dtype=np.float32,
    ).reshape(-1)

    file.write(
        f"static const float {name}[{len(values)}] = {{\n"
    )

    for start in range(
        0,
        len(values),
        values_per_line,
    ):
        line_values = values[
            start:start + values_per_line
        ]

        line = ", ".join(
            f"{float(value):.9g}f"
            for value in line_values
        )

        file.write(f"    {line},\n")

    file.write("};\n\n")


with open(
    LOF_HEADER_PATH,
    "w",
    encoding="utf-8",
) as file:
    file.write("#ifndef PURE_NORMAL_LOF_MODEL_H\n")
    file.write("#define PURE_NORMAL_LOF_MODEL_H\n\n")
    file.write("#include <math.h>\n")
    file.write("#include <stdint.h>\n\n")

    file.write(f"#define LOF_N_FEATURES {N_FEATURES}\n")
    file.write(f"#define LOF_WINDOW_LEN {WINDOW_LEN}\n")
    file.write(f"#define LOF_N_INPUTS {number_inputs}\n")
    file.write(
        f"#define LOF_N_TRAINING_WINDOWS "
        f"{number_training_windows}\n"
    )
    file.write(
        f"#define LOF_N_NEIGHBORS "
        f"{number_neighbors}\n"
    )
    file.write(
        f"#define LOF_SCALE_DIVISOR "
        f"{SCALE_DIVISOR:.1f}f\n"
    )
    file.write(
        f"#define LOF_SCORE_THRESHOLD "
        f"{FINAL_SCORE_THRESHOLD:.9g}f\n\n"
    )

    write_float_array(
        file,
        "LOF_SCALER_MIN",
        scaler_min,
    )

    write_float_array(
        file,
        "LOF_SCALER_MAX",
        scaler_max,
    )

    write_float_array(
        file,
        "LOF_TRAINING_WINDOWS",
        training_windows,
    )

    write_float_array(
        file,
        "LOF_TRAINING_LRD",
        training_lrd,
    )

    write_float_array(
        file,
        "LOF_TRAINING_K_DISTANCE",
        training_k_distance,
    )

    file.write(r'''
static inline float lof_squared_distance(
    const float *a,
    const float *b)
{
    float distance = 0.0f;

    for (int i = 0; i < LOF_N_INPUTS; i++)
    {
        float difference = a[i] - b[i];
        distance += difference * difference;
    }

    return distance;
}


static inline void lof_insert_neighbor(
    float distance,
    int index,
    float nearest_distance[LOF_N_NEIGHBORS],
    int nearest_index[LOF_N_NEIGHBORS])
{
    if (distance >= nearest_distance[LOF_N_NEIGHBORS - 1])
    {
        return;
    }

    int position = LOF_N_NEIGHBORS - 1;

    while (
        position > 0
        && distance < nearest_distance[position - 1]
    )
    {
        nearest_distance[position] =
            nearest_distance[position - 1];

        nearest_index[position] =
            nearest_index[position - 1];

        position--;
    }

    nearest_distance[position] = distance;
    nearest_index[position] = index;
}


static inline int lof_predict(
    const float raw_window[LOF_WINDOW_LEN][LOF_N_FEATURES],
    float *anomaly_score)
{
    float input[LOF_N_INPUTS];

    int input_index = 0;

    for (int time = 0; time < LOF_WINDOW_LEN; time++)
    {
        for (
            int feature = 0;
            feature < LOF_N_FEATURES;
            feature++
        )
        {
            float physical_value =
                raw_window[time][feature]
                / LOF_SCALE_DIVISOR;

            float range =
                LOF_SCALER_MAX[feature]
                - LOF_SCALER_MIN[feature];

            if (range > 0.0f)
            {
                input[input_index] =
                    (
                        physical_value
                        - LOF_SCALER_MIN[feature]
                    )
                    / range;
            }
            else
            {
                input[input_index] = 0.0f;
            }

            input_index++;
        }
    }

    float nearest_distance[LOF_N_NEIGHBORS];
    int nearest_index[LOF_N_NEIGHBORS];

    for (int i = 0; i < LOF_N_NEIGHBORS; i++)
    {
        nearest_distance[i] = INFINITY;
        nearest_index[i] = -1;
    }

    for (
        int sample = 0;
        sample < LOF_N_TRAINING_WINDOWS;
        sample++
    )
    {
        const float *training_sample =
            &LOF_TRAINING_WINDOWS[
                sample * LOF_N_INPUTS
            ];

        float squared_distance =
            lof_squared_distance(
                input,
                training_sample
            );

        lof_insert_neighbor(
            squared_distance,
            sample,
            nearest_distance,
            nearest_index
        );
    }

    float reachability_sum = 0.0f;

    for (int i = 0; i < LOF_N_NEIGHBORS; i++)
    {
        int neighbor = nearest_index[i];

        float distance = sqrtf(
            nearest_distance[i]
        );

        float reachability_distance =
            fmaxf(
                distance,
                LOF_TRAINING_K_DISTANCE[neighbor]
            );

        reachability_sum +=
            reachability_distance;
    }

    float query_lrd =
        1.0f
        / (
            reachability_sum
            / LOF_N_NEIGHBORS
            + 1.0e-10f
        );

    float lof_sum = 0.0f;

    for (int i = 0; i < LOF_N_NEIGHBORS; i++)
    {
        int neighbor = nearest_index[i];

        lof_sum +=
            LOF_TRAINING_LRD[neighbor]
            / query_lrd;
    }

    *anomaly_score =
        lof_sum
        / LOF_N_NEIGHBORS;

    return (
        *anomaly_score
        > LOF_SCORE_THRESHOLD
    );
}

#endif
''')

print("Saved:", LOF_HEADER_PATH)
print(
    "Header size MB:",
    round(
        os.path.getsize(LOF_HEADER_PATH)
        / (1024 * 1024),
        2,
    ),
)
print(
    "Use lof_predict(raw_window, &score)",
    "in your embedded C code.",
)

In [ ]:
# ==========================================================
# CELL 15 — manual LOF test
# ==========================================================
def manual_lof_prediction(test_file):
    model = joblib.load(MODEL_PATH)
    scaler = joblib.load(SCALER_PATH)
    thresholds = np.load(THRESHOLD_PATH)

    score_threshold = float(
        thresholds["score_threshold"]
    )

    file_threshold = float(
        thresholds[
            "file_anomaly_fraction_threshold"
        ]
    )

    test_df = read_one(test_file)

    X_test, starts = make_windows_from_array(
        test_df.to_numpy(dtype=np.float32),
        scaler,
    )

    scores = anomaly_score(
        model,
        X_test,
    )

    flags = (
        scores > score_threshold
    ).astype(np.int64)

    anomaly_fraction = float(
        np.mean(flags)
    )

    file_decision = (
        "ANOMALY"
        if anomaly_fraction >= file_threshold
        else "HEALTHY"
    )

    result = pd.DataFrame({
        "window": np.arange(len(X_test)),
        "start_row": starts,
        "time_sec": (
            starts + WINDOW_LEN / 2.0
        ) * ROW_PERIOD_SEC,
        "anomaly_score": scores,
        "score_threshold": score_threshold,
        "decision": np.where(
            flags == 1,
            "anomaly",
            "normal",
        ),
        "is_anomaly": flags,
    })

    stem = Path(test_file).stem

    output_csv = os.path.join(
        MANUAL_TEST_DIR,
        f"manual_test_{stem}.csv",
    )

    result.to_csv(
        output_csv,
        index=False,
    )

    print("\n" + "=" * 60)
    print("MANUAL LOF TEST")
    print("=" * 60)
    print("File:", test_file)
    print("Total windows:", len(result))
    print("Normal windows:", int(np.sum(flags == 0)))
    print("Anomalous windows:", int(np.sum(flags == 1)))
    print("Anomaly fraction:", anomaly_fraction)
    print("File threshold:", file_threshold)
    print("FILE DECISION:", file_decision)
    print("Score threshold:", score_threshold)
    print("Saved CSV:", output_csv)

    print("\nWindow decisions:")
    print(result["decision"].value_counts())

    plt.figure(figsize=(12, 4))
    plt.plot(
        result["time_sec"],
        result["anomaly_score"],
    )
    plt.axhline(
        score_threshold,
        linestyle="--",
        label="score threshold",
    )
    plt.xlabel("Time [s]")
    plt.ylabel("LOF anomaly score")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        os.path.join(
            MANUAL_TEST_DIR,
            f"manual_test_{stem}_score.png",
        ),
        dpi=150,
    )
    plt.show()

    plt.figure(figsize=(12, 4))
    plt.step(
        result["time_sec"],
        result["is_anomaly"],
        where="mid",
    )
    plt.yticks(
        [0, 1],
        ["normal", "anomaly"],
    )
    plt.xlabel("Time [s]")
    plt.ylabel("Decision")
    plt.tight_layout()
    plt.savefig(
        os.path.join(
            MANUAL_TEST_DIR,
            f"manual_test_{stem}_timeline.png",
        ),
        dpi=150,
    )
    plt.show()

    return result


In [ ]:
# ==========================================================
# CELL 16 — select manual test file
# ==========================================================
if __name__ == "__main__":
    import tkinter as tk
    from tkinter import filedialog

    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    TEST_FILE = filedialog.askopenfilename(
        title="Select motor file",
        filetypes=[
            ("Excel files", "*.xlsx"),
            ("CSV files", "*.csv"),
            ("All files", "*.*"),
        ],
    )

    root.destroy()

    if TEST_FILE:
        manual_result = manual_lof_prediction(
            TEST_FILE
        )
    else:
        print("No file selected.")